# Context and follow-up testing

Диагностика `request_scope`, `effective_request`, недавнего полного диалога и центральных catalog objects. Notebook использует отдельную SQLite-базу и реальный настроенный LLM только при выполнении ячеек.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sommelier.agent.graph import run_agent_turn
from sommelier.agent.resolver import build_recent_objects
from sommelier.agent.state import AgentState
from sommelier.storage.session_repository import SessionRepository

DB_PATH = ROOT / "data" / "notebook_context_followup.sqlite3"
SESSION_ID = "notebook-context-followup-v1"
repository = SessionRepository(DB_PATH)
print(DB_PATH)

c:\Users\egorg\Documents\SPIRT_TEST\data\notebook_context_followup.sqlite3


In [2]:
def reset_session(session_id: str = SESSION_ID) -> None:
    repository.delete_session(session_id)
    print(f"Cleared session: {session_id}")


def ask(message: str, session_id: str = SESSION_ID):
    memory_before = repository.load_session_memory(session_id)
    recent_dialogue = repository.load_recent_messages(session_id, limit=4)
    recent_objects = build_recent_objects(memory_before)
    previous_scope = (
        memory_before.turns[-1].request_scope
        if memory_before.turns else None
    )

    state = run_agent_turn(
        AgentState(session_id=session_id, user_request=message),
        config={"configurable": {"repository": repository}},
    )
    resolution = state.turn_resolution
    saved_memory = repository.load_session_memory(session_id)
    saved_turn = saved_memory.turns[-1] if saved_memory.turns else None

    print("\n" + "=" * 96)
    print("USER:", message)
    print("PREVIOUS SCOPE:", previous_scope)
    print("RECENT DIALOGUE BEFORE TURN:")
    for item in recent_dialogue:
        print(f"  {item['role']}: {item['content']}")
    print("RECENT OBJECTS BEFORE TURN:", recent_objects)
    print("FOLLOW UP:", resolution.follow_up if resolution else None)
    print("REQUEST SCOPE:", resolution.request_scope if resolution else None)
    print("INITIAL REQUEST:", resolution.initial_request if resolution else None)
    print("EFFECTIVE REQUEST:", resolution.effective_request if resolution else None)
    print("NEGATIVE REQUEST:", resolution.negative_request if resolution else None)
    print("REASONING NOTE:", resolution.reasoning_note if resolution else None)
    print("TOOL CALLS:", state.tool_call_count)
    print("TOOLS USED:")
    tool_events = [
        trace for trace in state.tool_traces
        if trace.tool_name == "tool_call"
    ]
    if not tool_events:
        print("  (none)")
    for trace in tool_events:
        tool_name = trace.input.get("tool", "unknown")
        tool_args = trace.input.get("args", {})
        print(
            f"  {tool_name}({tool_args}) -> "
            f"{trace.status}: {trace.output_summary}"
        )
    print("FULL CARDS THIS TURN:", [(card.kind, card.id) for card in state.cards])
    used = state.final_answer_result.used_result_refs if state.final_answer_result else []
    print("USED REFS:", [ref.model_dump(mode="json") for ref in used])
    print("ASSISTANT:", state.final_answer_result.answer if state.final_answer_result else None)
    print("ERRORS:", state.errors)
    print("SAVED TURN:", saved_turn.model_dump(mode="json") if saved_turn else None)
    return state


reset_session()

Cleared session: notebook-context-followup-v1


## Основной переход: food pairing → product → cocktails → sweet cocktails → recipe

In [3]:
dialog = [
    "Привет, хочу ром к жареному мясу.",
    "Нормально, а какие коктейли можно из него сделать?",
    "Давай сладкие варианты.",
    "Как приготовить первый из последних вариантов?",
]

states = [ask(message) for message in dialog]


USER: Привет, хочу ром к жареному мясу.
PREVIOUS SCOPE: None
RECENT DIALOGUE BEFORE TURN:
RECENT OBJECTS BEFORE TURN: []
FOLLOW UP: False
REQUEST SCOPE: food_pairing
INITIAL REQUEST: Привет, хочу ром к жареному мясу.
EFFECTIVE REQUEST: Подобрать ром к жареному мясу.
NEGATIVE REQUEST: None
REASONING NOTE: Standalone food pairing request for rum with fried/roasted meat.
TOOL CALLS: 1
TOOL TRACE:
  load_memory_and_profile: success — memory and profile loaded
  resolve_turn: success — follow_up=False
  tool_call: success — returned_after_filter=7; cards_in_current_turn=7
  generate_answer: success — used_refs=1
  persist: success — turn, transcript, profile and traces saved
FULL CARDS THIS TURN: [('product', 'bacardi-spiced-rum'), ('product', 'bacardi-raspberry-rum'), ('product', 'bacardi-carta-negra'), ('product', 'bacardi-dragonberry-rum'), ('product', 'bacardi-limon-rum'), ('product', 'bacardi-tropical'), ('product', 'bacardi-gran-reserva-limitada')]
USED REFS: [{'kind': 'product', 'id

Ожидание: scopes `food_pairing → cocktail → cocktail → recipe`; во втором и третьем `effective_request` должно сохраняться точное название последнего показанного рома.

## Profile update не сбрасывает последний центральный объект

In [4]:
PROFILE_SESSION_ID = "notebook-context-profile-gap-v1"
reset_session(PROFILE_SESSION_ID)

profile_gap_dialog = [
    "Посоветуй ром с кокосовым вкусом.",
    "Понятно. Запомни, что я не люблю горькое.",
    "Теперь посоветуй пряный ром к жареному мясу.",
    "А какие коктейли из него можно сделать?",
    "Давай сладкие варианты.",
]

profile_states = [
    ask(message, PROFILE_SESSION_ID)
    for message in profile_gap_dialog
]

Cleared session: notebook-context-profile-gap-v1

USER: Посоветуй ром с кокосовым вкусом.
PREVIOUS SCOPE: None
RECENT DIALOGUE BEFORE TURN:
RECENT OBJECTS BEFORE TURN: []
FOLLOW UP: False
REQUEST SCOPE: product
INITIAL REQUEST: Посоветуй ром с кокосовым вкусом.
EFFECTIVE REQUEST: Посоветовать ром с кокосовым вкусом.
NEGATIVE REQUEST: None
REASONING NOTE: Standalone product recommendation for a rum with coconut flavor.
TOOL CALLS: 2
TOOL TRACE:
  load_memory_and_profile: success — memory and profile loaded
  resolve_turn: success — follow_up=False
  tool_call: error — tool_execution_failed
  tool_call: error — tool_execution_failed
  generate_answer: success — used_refs=0
  persist: success — turn, transcript, profile and traces saved
FULL CARDS THIS TURN: []
USED REFS: []
ASSISTANT: Похоже, сейчас у меня нет доступных вариантов из каталога, чтобы уверенно посоветовать конкретный ром с кокосовым вкусом. Могу помочь точнее, если скажете: вам нужен скорее сладкий кокосовый ром или более с

## Коррекция объекта сохраняет незавершённое действие

In [5]:
CORRECTION_SESSION_ID = "notebook-context-correction-v1"
reset_session(CORRECTION_SESSION_ID)

correction_dialog = [
    "Посоветуй BACARDÍ Coconut rum.",
    "Какие коктейли из него можно сделать?",
    "Нет, про BACARDÍ Spiced говорю — подбери коктейли из него.",
    "Давай сладкие варианты.",
]

correction_states = [
    ask(message, CORRECTION_SESSION_ID)
    for message in correction_dialog
]

Cleared session: notebook-context-correction-v1

USER: Посоветуй BACARDÍ Coconut rum.
PREVIOUS SCOPE: None
RECENT DIALOGUE BEFORE TURN:
RECENT OBJECTS BEFORE TURN: []
FOLLOW UP: False
REQUEST SCOPE: product
INITIAL REQUEST: Посоветуй BACARDÍ Coconut rum.
EFFECTIVE REQUEST: Посоветовать ром BACARDÍ Coconut rum.
NEGATIVE REQUEST: None
REASONING NOTE: Explicit standalone product recommendation; no prior context.
TOOL CALLS: 1
TOOL TRACE:
  load_memory_and_profile: success — memory and profile loaded
  resolve_turn: success — follow_up=False
  tool_call: error — tool_execution_failed
  generate_answer: success — used_refs=0
  persist: success — turn, transcript, profile and traces saved
FULL CARDS THIS TURN: []
USED REFS: []
ASSISTANT: Конечно, думаю, вам подойдёт BACARDÍ Coconut rum, если хочется мягкий, тропический ром без лишней тяжести. У него обычно читаются кокосовая сладость, лёгкий сливочный оттенок и очень мягкий, дружелюбный профиль — такой вариант удобно пить в коктейлях с анана

KeyboardInterrupt: 

## Независимый запрос должен сбросить follow-up

In [ ]:
independent_state = ask(
    "Теперь посоветуй выдержанный ром для Old Fashioned.",
    SESSION_ID,
)
assert independent_state.turn_resolution.follow_up is False
assert independent_state.turn_resolution.request_scope == "product"

## Полное сохранённое состояние

In [ ]:
memory = repository.load_session_memory(SESSION_ID)
profile = repository.load_user_profile(SESSION_ID)
messages = repository.load_messages(SESSION_ID)
traces = repository.load_trace_events(SESSION_ID)

print("MEMORY:", memory.model_dump(mode="json"))
print("PROFILE:", profile.model_dump(mode="json"))
print("MESSAGES:", messages)
print("TRACE COUNT:", len(traces))